# Spark + Kafka CLUSTER (Worker, versi Performa): Atlas Iklim Kota-kota Indonesia
**Big Data & Predictive Analytics** · dijalankan di **PC Mahasiswa** (Worker 1 sampai 5)

Isi `KELOMPOK`, `NOMOR_WORKER` (1–5), `MASTER_IP` di Sel 1. Jalankan Sel 1 → 8; **Sel 8 terus berjalan** dan melayani tugas kasus nyata **serta perintah benchmark** dari Master (jangan dihentikan sampai Master selesai).

**Baru:** `PemantauResource` (CPU, RAM, jaringan, disk lewat `psutil`), stempel waktu tiap tahap tugas, serta Sel 7 untuk benchmark partisi × Worker dan mikrobenchmark Kafka.
Jika internet bermasalah: `MODE_SIMULASI = True` di Sel 1. Pasang `pip install psutil` agar resource terukur.

## Sel 1. Konfigurasi, Kafka, pemantau resource, dan pelapor

In [ ]:
# ===== SEL 1: Konfigurasi, Kafka, pemantau resource, dan fungsi pelapor (PC Mahasiswa / Worker) =====
# !pip install confluent-kafka pyspark pandas matplotlib seaborn psutil      # jalankan sekali jika belum terpasang
import os, sys, io, json, time, glob, socket, zlib, contextlib, threading, uuid
import urllib.request, urllib.parse, urllib.error
from pathlib import Path

from confluent_kafka import Producer, Consumer, KafkaError, KafkaException

# ---------------- ISI SESUAI KELOMPOK ANDA ----------------
KELOMPOK     = "k1"               # k1 ... k8  (HARUS sama dengan Master kelompok Anda)
NOMOR_WORKER = 1                  # 1 ... 5
MASTER_IP    = "10.73.3.90"       # IP PC Master kelompok (broker Kafka berjalan di sini)
# -----------------------------------------------------------

BOOTSTRAP = f"{MASTER_IP}:9092"
TOPIC_TUGAS  = f"{KELOMPOK}-tugas"       # Master -> Worker : kota yang harus diunduh
TOPIC_HASIL  = f"{KELOMPOK}-hasil"       # Worker -> Master : ringkasan hasil olahan
TOPIC_STATUS = f"{KELOMPOK}-status"      # Worker -> Master : progres & kesiapan
TOPIC_KONTROL = f"{KELOMPOK}-kontrol"    # Master -> Worker : perintah benchmark
TOPIC_BENCH  = f"{KELOMPOK}-bench"       # Worker -> Master : hasil benchmark
GROUP_ID     = f"{KELOMPOK}-pekerja"     # semua Worker satu kelompok = satu consumer group
HOST = f"{socket.gethostname()}-w{NOMOR_WORKER}"

MODE_SIMULASI    = False          # True jika internet kampus bermasalah (data cuaca tiruan)
FAKTOR_PERBESAR  = 8              # data diperbanyak N kali untuk percobaan performa
MAKS_UMUR_TUGAS  = None           # detik; mis. 7200 = lewati tugas sisa percobaan lama

DATA_DIR = Path("data") / KELOMPOK        # data mentah (CSV) hasil unduhan
OUT_DIR  = Path("output") / KELOMPOK      # gambar EDA
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
BENCH_DIR = Path("data") / f"{KELOMPOK}-bench"   # berkas sementara benchmark: JANGAN di DATA_DIR (akan ikut terbaca percobaan Spark)
BENCH_DIR.mkdir(parents=True, exist_ok=True)
for _sisa in DATA_DIR.glob("bench_*.csv"):         # sisa versi lama yang tercampur dengan CSV kota
    try:
        _sisa.unlink()
    except OSError:
        pass
RUN = "-"                                 # diisi dari pesan tugas

try:
    import psutil
except ImportError:
    psutil = None          # pip install psutil  (tanpa psutil, pengukuran resource dilewati)

class PemantauResource:
    """Mencuplik CPU, RAM, jaringan, dan disk di thread terpisah selama sebuah blok pekerjaan.
    Pakai: pm = PemantauResource().mulai(); ...pekerjaan...; ringkasan = pm.berhenti()"""
    def __init__(self, interval=0.5):
        self.interval = interval
        self._henti = threading.Event()
        self._t = None
        self.sampel = []

    def _pohon(self):
        p = psutil.Process()
        try:
            return [p] + p.children(recursive=True)        # termasuk proses Java milik PySpark
        except psutil.Error:
            return [p]

    def _cpu_detik(self):
        total = 0.0
        for pr in self._pohon():
            try:
                c = pr.cpu_times(); total += c.user + c.system
            except psutil.Error:
                pass
        return total

    def _rss_mb(self):
        total = 0
        for pr in self._pohon():
            try:
                total += pr.memory_info().rss
            except psutil.Error:
                pass
        return total / 1e6

    def mulai(self):
        if psutil is None:
            return self
        self._t0 = time.perf_counter()
        self._cpu0 = self._cpu_detik()
        self._net0 = psutil.net_io_counters()
        try:
            self._disk0 = psutil.disk_io_counters()
        except Exception:
            self._disk0 = None
        psutil.cpu_percent(None)                            # inisialisasi pembanding
        self._henti.clear(); self.sampel = []
        self._t = threading.Thread(target=self._loop, daemon=True); self._t.start()
        return self

    def _loop(self):
        while not self._henti.wait(self.interval):
            self.sampel.append((psutil.cpu_percent(None), self._rss_mb(), psutil.virtual_memory().percent))

    def berhenti(self):
        if psutil is None:
            return {"psutil": False}
        self._henti.set()
        if self._t:
            self._t.join(2)
        durasi = max(time.perf_counter() - self._t0, 1e-9)
        net1 = psutil.net_io_counters()
        try:
            d1 = psutil.disk_io_counters()
        except Exception:
            d1 = None
        cpu = [s[0] for s in self.sampel] or [psutil.cpu_percent(None)]
        ram = [s[1] for s in self.sampel] or [self._rss_mb()]
        rams = [s[2] for s in self.sampel] or [psutil.virtual_memory().percent]
        cpu_detik = max(0.0, self._cpu_detik() - self._cpu0)
        return {"psutil": True, "durasi": round(durasi, 3),
                "cpu_sistem_rata": round(sum(cpu) / len(cpu), 1), "cpu_sistem_maks": round(max(cpu), 1),
                "cpu_proses_core": round(cpu_detik / durasi, 2),          # rata-rata core yang dipakai proses ini
                "ram_proses_maks_mb": round(max(ram), 1), "ram_sistem_maks_persen": round(max(rams), 1),
                "net_tx_mb": round((net1.bytes_sent - self._net0.bytes_sent) / 1e6, 3),
                "net_rx_mb": round((net1.bytes_recv - self._net0.bytes_recv) / 1e6, 3),
                "disk_tulis_mb": round((d1.write_bytes - self._disk0.write_bytes) / 1e6, 3) if d1 and self._disk0 else None,
                "jumlah_sampel": len(self.sampel)}

produser = Producer({
    "bootstrap.servers": BOOTSTRAP,
    "acks": "all",
    "compression.type": "gzip",
    "linger.ms": 50,
})

galat_kirim = []
def _hasil_kirim(err, msg):
    if err is not None:
        galat_kirim.append(str(err))

def kirim(topic, key, obj):
    produser.produce(topic, key=key, value=json.dumps(obj).encode("utf-8"), on_delivery=_hasil_kirim)
    produser.poll(0)

def lapor(tugas, status, **tambahan):
    kirim(TOPIC_STATUS, HOST, {"run": RUN, "worker": HOST, "tugas": tugas["tugas"],
                               "status": status, "waktu": time.time(), **tambahan})

def lapor_siap(consumer, partisi):
    """Dipanggil Kafka saat Worker ini mendapat jatah partisi."""
    kirim(TOPIC_STATUS, HOST, {"run": "-", "worker": HOST, "status": "SIAP", "waktu": time.time(),
                               "partisi": sorted(p.partition for p in partisi)})
    produser.flush()
    print("Partisi tugas yang dipegang:", sorted(p.partition for p in partisi))

meta = produser.list_topics(timeout=5)
print("Terhubung ke Kafka. Topic kelompok:", sorted(t for t in meta.topics if t.startswith(KELOMPOK)))
print("Nama Worker :", HOST, "| Kelompok:", KELOMPOK)

## Sel 2. Spark lokal: skema, kueri SQL, dan alat ukur

In [ ]:
# ===== SEL 2: Spark lokal (DataFrame & SQL), skema, dan kueri =====
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

spark = (SparkSession.builder
         .appName(f"{KELOMPOK}-worker{NOMOR_WORKER}")
         .master("local[2]")                                  # Worker mengolah datanya sendiri secara lokal
         .config("spark.driver.memory", "1g")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.sql.shuffle.partitions", "8")         # data kecil: 8 sudah cukup (bawaan 200)
         .config("spark.pyspark.python", sys.executable)
         .config("spark.pyspark.driver.python", sys.executable)
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| master:", spark.sparkContext.master, "| UI:", spark.sparkContext.uiWebUrl)

# Skema eksplisit: lebih cepat dan lebih aman daripada inferSchema
SKEMA = StructType([
    StructField("waktu",      TimestampType(), True),
    StructField("suhu",       DoubleType(),    True),
    StructField("kelembapan", DoubleType(),    True),
    StructField("hujan",      DoubleType(),    True),
    StructField("angin",      DoubleType(),    True),
    StructField("kota",       StringType(),    True),
])
FORMAT_WAKTU = "yyyy-MM-dd HH:mm"

# Kueri SQL (view 'cuaca' dibuat dari DataFrame bersih; view 'harian' dari hasil SQL_HARIAN)
SQL_HARIAN = """
SELECT kota, tahun, bulan, tanggal,
       AVG(suhu)       AS suhu,
       MAX(suhu)       AS suhu_maks,
       MIN(suhu)       AS suhu_min,
       SUM(hujan)      AS hujan,
       AVG(kelembapan) AS kelembapan,
       AVG(angin)      AS angin
FROM cuaca
GROUP BY kota, tahun, bulan, tanggal
"""
SQL_BULANAN = """
SELECT kota, tahun, bulan,
       ROUND(AVG(suhu), 2)                               AS suhu_rata,
       ROUND(MAX(suhu_maks), 1)                          AS suhu_maks,
       ROUND(MIN(suhu_min), 1)                           AS suhu_min,
       ROUND(SUM(hujan), 1)                              AS hujan_total,
       SUM(CASE WHEN hujan >= 1 THEN 1 ELSE 0 END)      AS hari_hujan,
       ROUND(AVG(kelembapan), 1)                         AS kelembapan_rata,
       ROUND(AVG(angin), 1)                              AS angin_rata,
       COUNT(*)                                          AS jumlah_hari
FROM harian
GROUP BY kota, tahun, bulan
"""
SQL_HARI_TERBASAH = """
SELECT t.tahun, t.tanggal, ROUND(t.hujan, 1) AS hujan_mm, t.rn AS peringkat
FROM (SELECT tahun, tanggal, hujan,
             ROW_NUMBER() OVER (PARTITION BY tahun ORDER BY hujan DESC) AS rn
      FROM harian) t
WHERE t.rn <= 3
ORDER BY t.tahun, t.rn
"""

def ukur(fn, ulang=2):
    """Jalankan fn beberapa kali, kembalikan waktu terbaik (detik)."""
    terbaik = None
    for _ in range(ulang):
        t0 = time.time(); fn(); d = time.time() - t0
        terbaik = d if terbaik is None else min(terbaik, d)
    return round(terbaik, 3)

def rencana(df):
    """Tangkap keluaran df.explain() sebagai teks."""
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            df.explain()
    except Exception:
        pass
    return buf.getvalue()

def jalankan_penuh(df):
    """Paksa seluruh perhitungan berjalan tanpa mengumpulkan hasil (untuk pengukuran)."""
    df.write.format("noop").mode("overwrite").save()

## Sel 3. Data Ingestion: mengunduh data cuaca

In [ ]:
# ===== SEL 3: Data Ingestion - mengunduh data cuaca (API Open-Meteo) =====
import numpy as np
import pandas as pd

URL_API   = "https://archive-api.open-meteo.com/v1/archive"
VARIABEL  = "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m"

def parse_open_meteo(obj, kota):
    """Ubah JSON Open-Meteo menjadi DataFrame pandas (kolom sama dengan SKEMA)."""
    h = obj["hourly"]
    return pd.DataFrame({
        "waktu":      [t.replace("T", " ") for t in h["time"]],
        "suhu":       h["temperature_2m"],
        "kelembapan": h["relative_humidity_2m"],
        "hujan":      h["precipitation"],
        "angin":      h["wind_speed_10m"],
        "kota":       kota,
    })

def unduh_cuaca(kota, lat, lon, mulai, akhir, percobaan=4):
    """Unduh data per jam. Mengulang jika kena batas laju (HTTP 429) atau jaringan putus."""
    parameter = urllib.parse.urlencode({
        "latitude": lat, "longitude": lon, "start_date": mulai, "end_date": akhir,
        "hourly": VARIABEL, "timezone": "Asia/Jakarta"})
    req = urllib.request.Request(f"{URL_API}?{parameter}",
                                 headers={"User-Agent": "KuliahBigData/1.0 (dosen@kampus.ac.id)"})  # ganti kontak
    for k in range(1, percobaan + 1):
        try:
            with urllib.request.urlopen(req, timeout=90) as resp:
                return parse_open_meteo(json.loads(resp.read()), kota)
        except urllib.error.HTTPError as e:
            if e.code in (429, 500, 502, 503, 504) and k < percobaan:
                print(f"  HTTP {e.code}, coba lagi {k}/{percobaan - 1} ...")
                time.sleep(10 * k)
            else:
                raise
        except (urllib.error.URLError, TimeoutError, ConnectionError) as e:
            if k == percobaan:
                raise
            print(f"  Jaringan bermasalah ({e}), coba lagi {k}/{percobaan - 1} ...")
            time.sleep(5 * k)

def simulasi_cuaca(kota, lat, lon, mulai, akhir):
    """Data cuaca tiruan dengan pola nyata: siklus harian, musim hujan, sedikit data hilang."""
    rnd = np.random.default_rng(zlib.crc32(kota.encode()))
    waktu = pd.date_range(f"{mulai} 00:00", f"{akhir} 23:00", freq="h")
    n = len(waktu)
    jam, hari = waktu.hour.values, waktu.dayofyear.values
    tinggi = rnd.uniform(0.0, 1.0)                                  # 0 = pantai, 1 = dataran tinggi
    puncak = 15 if lat < 0 else 320                                 # puncak musim hujan (hari ke-)
    fase = np.cos((hari - puncak) / 365 * 2 * np.pi)                # -1 .. 1
    suhu = 28.5 - 4.5 * tinggi + 2.8 * np.sin((jam - 9) / 24 * 2 * np.pi) + rnd.normal(0, 0.6, n)
    p_hujan = np.clip(0.07 + 0.06 * fase + 0.03 * tinggi, 0.01, 0.4)
    hujan = np.where(rnd.random(n) < p_hujan, rnd.exponential(2.4, n), 0.0).round(1)
    kelembapan = np.clip(76 + 9 * fase + 6 * tinggi - 1.2 * (suhu - 27) + rnd.normal(0, 3, n), 30, 100).round(0)
    angin = np.abs(rnd.normal(9, 3, n) + 1.5 * (1 - fase)).round(1)
    suhu = suhu.round(1)
    suhu[rnd.random(n) < 0.002] = np.nan                            # data hilang untuk latihan pembersihan
    return pd.DataFrame({"waktu": waktu.strftime("%Y-%m-%d %H:%M"), "suhu": suhu,
                         "kelembapan": kelembapan, "hujan": hujan, "angin": angin, "kota": kota})

def ingest_kota(tugas):
    """Ambil data satu kota, simpan sebagai CSV mentah. Mengembalikan (path, jumlah_baris, detik)."""
    t0 = time.time()
    if MODE_SIMULASI:
        pdf = simulasi_cuaca(tugas["kota"], tugas["lat"], tugas["lon"], tugas["mulai"], tugas["akhir"])
    else:
        pdf = unduh_cuaca(tugas["kota"], tugas["lat"], tugas["lon"], tugas["mulai"], tugas["akhir"])
    path = DATA_DIR / f"{tugas['kota'].replace(' ', '_')}.csv"
    pdf.to_csv(path, index=False)                                   # lapisan "raw"
    return path, len(pdf), round(time.time() - t0, 2)

print("Fungsi ingestion siap. Mode simulasi:", MODE_SIMULASI)

## Sel 4. Pengolahan (DataFrame + SQL) dan EDA

In [ ]:
# ===== SEL 4: Pengolahan (DataFrame + SQL) dan EDA (matplotlib + seaborn) =====
import matplotlib.pyplot as plt
import seaborn as sns

def muat_dan_bersihkan(path):
    """Baca CSV dengan skema eksplisit, bersihkan, tambah kolom turunan, daftarkan view 'cuaca'."""
    mentah = (spark.read.option("header", "true")
                        .option("timestampFormat", FORMAT_WAKTU)
                        .schema(SKEMA).csv(Path(path).resolve().as_uri()))
    bersih = (mentah.dropDuplicates(["kota", "waktu"])                 # buang baris dobel
                    .dropna(subset=["waktu", "suhu"])                  # buang baris tanpa waktu/suhu
                    .fillna({"hujan": 0.0})                            # hujan kosong dianggap 0
                    .withColumn("tanggal", F.to_date("waktu"))
                    .withColumn("tahun", F.year("waktu"))
                    .withColumn("bulan", F.month("waktu")))
    bersih.createOrReplaceTempView("cuaca")
    return mentah, bersih

def buat_eda(harian, kota, path, tampil=True):
    """Satu gambar EDA 2x2 dari data harian (pandas)."""
    sns.set_theme(style="whitegrid")
    d = harian.copy()
    d["tanggal"] = pd.to_datetime(d["tanggal"])
    d = d.sort_values("tanggal")
    d["tahun"], d["bulan"] = d["tanggal"].dt.year, d["tanggal"].dt.month
    per_bulan = d.groupby(["tahun", "bulan"], as_index=False)["hujan"].sum()
    versi = tuple(int(x) for x in sns.__version__.split(".")[:2])
    galat = {"errorbar": "sd"} if versi >= (0, 12) else {"ci": "sd"}

    fig, ax = plt.subplots(2, 2, figsize=(13, 8))
    ax[0, 0].plot(d["tanggal"], d["suhu"], lw=0.6, alpha=0.5, label="harian")
    ax[0, 0].plot(d["tanggal"], d["suhu"].rolling(30, min_periods=7).mean(), lw=2, label="rata-rata 30 hari")
    ax[0, 0].set(title="Suhu harian (\u00b0C)", xlabel="", ylabel="\u00b0C"); ax[0, 0].legend()
    sns.barplot(data=per_bulan, x="bulan", y="hujan", ax=ax[0, 1], color="#4C78A8", **galat)
    ax[0, 1].set(title="Curah hujan per bulan (rata-rata antartahun \u00b1 sd)", xlabel="bulan", ylabel="mm")
    sns.histplot(d["suhu"], kde=True, ax=ax[1, 0], color="#F58518")
    ax[1, 0].set(title="Sebaran suhu harian", xlabel="\u00b0C")
    pivot = per_bulan.pivot(index="tahun", columns="bulan", values="hujan")
    sns.heatmap(pivot, annot=True, fmt=".0f", cmap="Blues", ax=ax[1, 1], cbar=False)
    ax[1, 1].set(title="Hujan total (mm): tahun \u00d7 bulan", xlabel="bulan", ylabel="")
    fig.suptitle(f"EDA cuaca {kota}", fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    if tampil:
        plt.show()
    plt.close(fig)

def proses_kota(path, kota, tampil=True):
    """Pipeline lengkap satu kota. Mengembalikan daftar rekaman bulanan (dict) untuk dikirim ke Master."""
    mentah, bersih = muat_dan_bersihkan(path)
    n_mentah, n_bersih = mentah.count(), bersih.count()
    print(f"  [{kota}] baris mentah={n_mentah:,} | setelah dibersihkan={n_bersih:,} | dibuang={n_mentah - n_bersih:,}")

    spark.sql(SQL_HARIAN).createOrReplaceTempView("harian")             # SQL
    bulanan_sql = spark.sql(SQL_BULANAN).orderBy("tahun", "bulan")

    # Versi DataFrame API (hasilnya harus sama dengan SQL)
    harian_df = (bersih.groupBy("kota", "tahun", "bulan", "tanggal")
                 .agg(F.avg("suhu").alias("suhu"), F.max("suhu").alias("suhu_maks"),
                      F.min("suhu").alias("suhu_min"), F.sum("hujan").alias("hujan"),
                      F.avg("kelembapan").alias("kelembapan"), F.avg("angin").alias("angin")))
    bulanan_df = (harian_df.groupBy("kota", "tahun", "bulan")
                  .agg(F.round(F.avg("suhu"), 2).alias("suhu_rata"),
                       F.round(F.max("suhu_maks"), 1).alias("suhu_maks"),
                       F.round(F.min("suhu_min"), 1).alias("suhu_min"),
                       F.round(F.sum("hujan"), 1).alias("hujan_total"),
                       F.sum(F.when(F.col("hujan") >= 1, 1).otherwise(0)).alias("hari_hujan"),
                       F.round(F.avg("kelembapan"), 1).alias("kelembapan_rata"),
                       F.round(F.avg("angin"), 1).alias("angin_rata"),
                       F.count(F.lit(1)).alias("jumlah_hari")))
    selisih = bulanan_sql.select(*bulanan_df.columns).exceptAll(bulanan_df).count()
    print(f"  [{kota}] SQL vs DataFrame API: {'SAMA' if selisih == 0 else str(selisih) + ' baris berbeda'}")

    top = spark.sql(SQL_HARI_TERBASAH).toPandas()                       # window function
    print(f"  [{kota}] hari terbasah per tahun:\n{top.to_string(index=False)}")

    harian_pdf = spark.sql("SELECT * FROM harian").toPandas()           # hanya ~1.000 baris -> aman
    buat_eda(harian_pdf, kota, OUT_DIR / f"{kota.replace(' ', '_')}_eda.png", tampil)
    return [r.asDict() for r in bulanan_sql.collect()]

print("Fungsi pengolahan & EDA siap.")

## Sel 5. Percobaan Spark: partition, shuffle, cache, format file, join

In [ ]:
# ===== SEL 5: Percobaan Partition, Shuffle & Performance (data milik Worker sendiri) =====
REF_KECIL = [("Jakarta", "Jawa"), ("Bandung", "Jawa"), ("Medan", "Sumatera")]   # contoh tabel kecil untuk join

def jalankan_performa(tugas, lapor_tahap=lambda t: None):
    """Mengukur waktu beberapa percobaan. Mengembalikan daftar rekaman {eksperimen, varian, detik, catatan}."""
    hasil = []
    def catat(eks, var, detik, cat=""):
        hasil.append({"eksperimen": eks, "varian": str(var), "detik": detik, "catatan": cat})
        print(f"  {eks:<26} {str(var):<22} {detik:>8.3f} dtk {cat}")

    berkas = list(DATA_DIR.glob("*.csv"))
    if not berkas:
        raise RuntimeError("tidak ada CSV lokal; Worker ini belum mengunduh kota apa pun")
    df = (spark.read.option("header", "true").option("timestampFormat", FORMAT_WAKTU).schema(SKEMA)
          .csv(DATA_DIR.resolve().as_uri() + "/*.csv")
          .dropna(subset=["waktu", "suhu"]).fillna({"hujan": 0.0})
          .withColumn("tahun", F.year("waktu")).withColumn("bulan", F.month("waktu")))
    besar = df.withColumn("salinan", F.explode(F.array(*[F.lit(i) for i in range(FAKTOR_PERBESAR)])))
    print(f"  Data dasar: {df.count():,} baris | diperbesar x{FAKTOR_PERBESAR}: {besar.count():,} baris")

    aqe_awal = spark.conf.get("spark.sql.adaptive.enabled", "true")
    shuf_awal = spark.conf.get("spark.sql.shuffle.partitions")
    bc_awal = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
    agregat = lambda d: d.groupBy("kota", "salinan", "tahun", "bulan").agg(F.avg("suhu").alias("s"), F.sum("hujan").alias("h"))

    # E1: format file (CSV vs Parquet)
    lapor_tahap("E1 format")
    try:
        catat("E1 format file", "CSV", ukur(lambda: jalankan_penuh(
            spark.read.option("header", "true").option("timestampFormat", FORMAT_WAKTU).schema(SKEMA)
                 .csv(DATA_DIR.resolve().as_uri() + "/*.csv").groupBy("kota").agg(F.avg("suhu")))))
        pq = (OUT_DIR / "parquet_tmp").resolve().as_uri()
        df.write.mode("overwrite").parquet(pq)
        catat("E1 format file", "Parquet", ukur(lambda: jalankan_penuh(
            spark.read.parquet(pq).groupBy("kota").agg(F.avg("suhu")))))
    except Exception as e:                       # Windows tanpa winutils.exe tidak bisa menulis Parquet
        catat("E1 format file", "Parquet(dilewati)", 0.0, str(e)[:60])

    # E2: jumlah partisi shuffle (AQE dimatikan agar pengaruhnya terlihat)
    lapor_tahap("E2 shuffle")
    spark.conf.set("spark.sql.adaptive.enabled", "false")
    for n in (200, 16, 4):
        spark.conf.set("spark.sql.shuffle.partitions", str(n))
        catat("E2 shuffle partitions", n, ukur(lambda: jalankan_penuh(agregat(besar))))
    spark.conf.set("spark.sql.shuffle.partitions", shuf_awal)
    spark.conf.set("spark.sql.adaptive.enabled", aqe_awal)

    # E3: jumlah partisi data (repartition)
    lapor_tahap("E3 partisi")
    for n in (1, 4, 16, 64):
        catat("E3 repartition", n, ukur(lambda: jalankan_penuh(agregat(besar.repartition(n)))),
              f"partisi={besar.repartition(n).rdd.getNumPartitions()}")

    # E4: cache
    lapor_tahap("E4 cache")
    agr = agregat(besar)
    dua_aksi = lambda: (agr.count(), agr.filter(F.col("h") > 100).count())
    catat("E4 cache", "tanpa cache", ukur(dua_aksi, ulang=1))
    agr.cache(); agr.count()
    catat("E4 cache", "dengan cache", ukur(dua_aksi, ulang=1))
    agr.unpersist()

    # E5: strategi join (tabel kecil)
    lapor_tahap("E5 join")
    ref = spark.createDataFrame(REF_KECIL, ["kota", "wilayah"])
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    j_sm = besar.join(ref, "kota").groupBy("wilayah").agg(F.avg("suhu"))
    catat("E5 join", "sort-merge", ukur(lambda: jalankan_penuh(j_sm)),
          "SortMergeJoin" if "SortMergeJoin" in rencana(j_sm) else "?")
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", bc_awal)
    j_bc = besar.join(F.broadcast(ref), "kota").groupBy("wilayah").agg(F.avg("suhu"))
    catat("E5 join", "broadcast", ukur(lambda: jalankan_penuh(j_bc)),
          "BroadcastHashJoin" if "BroadcastHashJoin" in rencana(j_bc) else "?")
    return hasil

print("Fungsi percobaan performa siap.")

## Sel 6. Fungsi pengerjaan tugas (dengan pengukuran latency & resource)
Setiap tugas mencatat `t_dibuat`, `t_terima`, `t_mulai`, `t_selesai`, durasi tiap tahap, dan ringkasan resource.

In [ ]:
# ===== SEL 6: Fungsi pengerjaan tugas (dengan pengukuran latency & resource) =====
def kerjakan_kota(tugas):
    path, n_baris, t_unduh = ingest_kota(tugas)                              # 1. ingestion
    print(f"  [{tugas['kota']}] unduh/simulasi {n_baris:,} baris dalam {t_unduh} dtk -> {path}")
    lapor(tugas, "MEMPROSES", baris=n_baris, detik_unduh=t_unduh)
    p0 = time.perf_counter()
    rekaman = proses_kota(path, tugas["kota"])                               # 2. DataFrame + SQL + EDA
    for r in rekaman:
        r["jenis"] = "bulanan"
    return rekaman, {"baris": n_baris, "detik_unduh": t_unduh, "detik_proses": round(time.perf_counter() - p0, 2)}

def kerjakan_performa(tugas):
    p0 = time.perf_counter()
    rekaman = jalankan_performa(tugas, lambda tahap: lapor(tugas, "MEMPROSES", tahap=tahap))
    for r in rekaman:
        r["jenis"] = "perf"
    return rekaman, {"baris": 0, "detik_unduh": 0, "detik_proses": round(time.perf_counter() - p0, 2)}

def kerjakan(tugas, t_terima=None):
    """t_terima = waktu pesan tugas diterima dari Kafka (untuk menghitung waktu antre)."""
    global RUN
    RUN = tugas["run"]
    t_terima = t_terima or time.time()
    pm = PemantauResource().mulai()                  # mulai mencuplik CPU / RAM / jaringan / disk
    t_mulai = time.time()
    lapor(tugas, "MENGUNDUH" if tugas["jenis"] == "kota" else "MEMPROSES")
    produser.flush()
    try:
        rekaman, info = kerjakan_kota(tugas) if tugas["jenis"] == "kota" else kerjakan_performa(tugas)
        lapor(tugas, "MENGIRIM", jumlah=len(rekaman))
        galat_kirim.clear()
        p0 = time.perf_counter()
        for r in rekaman:
            kunci = (f"{r['kota']}-{r['tahun']}-{r['bulan']}" if r["jenis"] == "bulanan"
                     else f"{r['eksperimen']}-{r['varian']}")
            kirim(TOPIC_HASIL, kunci, {**r, "run": RUN, "tugas": tugas["tugas"], "worker": HOST})
        sisa = produser.flush(60)
        info["detik_kirim"] = round(time.perf_counter() - p0, 3)
        if sisa > 0 or galat_kirim:
            raise RuntimeError(f"hasil belum terkirim (sisa={sisa}, galat={galat_kirim[:1]})")
        res = pm.berhenti()
        lapor(tugas, "SELESAI", jumlah=len(rekaman), t_dibuat=tugas.get("dibuat"), t_terima=t_terima,
              t_mulai=t_mulai, t_selesai=time.time(), res=res, **info)      # jumlah = janji; Master mencocokkannya
        produser.flush()
        print(f"Tugas {tugas['tugas']} ({tugas.get('kota', 'performa')}) selesai: {len(rekaman)} rekaman terkirim")
    except Exception as e:
        pm.berhenti()
        lapor(tugas, "GAGAL", galat=str(e)[:120])
        produser.flush()
        print(f"Tugas {tugas['tugas']} GAGAL:", e)

print("Fungsi pengerjaan tugas siap.")

## Sel 7. Benchmark: latency Kafka, throughput, dan resource (BARU)
`beban_kerja` adalah pekerjaan standar agar semua kombinasi partisi × Worker dibandingkan secara adil.

In [ ]:
# ===== SEL 7: Benchmark - latency Kafka, throughput, dan resource (diperintah Master) =====
import math, random

def persentil(data, q):
    s = sorted(data)
    if not s:
        return None
    k = (len(s) - 1) * q / 100
    f = int(k); c = min(f + 1, len(s) - 1)
    return s[f] + (s[c] - s[f]) * (k - f)

def beban_kerja(tugas):
    """Pekerjaan STANDAR benchmark: data cuaca 1 tahun, tulis & baca CSV (disk), lalu hitung CPU.
    'beban' = jumlah satuan CPU (1 satuan = 1 juta iterasi)."""
    pdf = simulasi_cuaca(tugas["kota"], tugas["lat"], tugas["lon"], "2022-01-01", "2022-12-31")
    tmp = BENCH_DIR / f"bench_{HOST}.csv"
    pdf.to_csv(tmp, index=False)                                   # beban disk
    d = pd.read_csv(tmp)
    x = 0.0
    for _ in range(max(1, int(tugas.get("beban", 3)))):            # beban CPU
        for i in range(1_000_000):
            x += math.sin(i)
    d["suhu_rata24"] = d["suhu"].rolling(24, min_periods=1).mean()
    return len(d)

def ikut_eksperimen(d, ctl):
    """Ikut satu percobaan: bergabung ke consumer group khusus, kerjakan tugas, ukur resource."""
    eid, topik = d["eid"], d["topik"]
    t_ctl = time.perf_counter()

    def saat_dapat_partisi(c, parts):
        kirim(TOPIC_BENCH, HOST, {"jenis": "join", "run": d["run"], "eid": eid, "worker": HOST, "nomor": NOMOR_WORKER,
                                  "partisi": sorted(p.partition for p in parts),
                                  "detik_join": round(time.perf_counter() - t_ctl, 3), "waktu": time.time()})

    tc = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": d["group"], "auto.offset.reset": "earliest",
                   "enable.auto.commit": False, "max.poll.interval.ms": 300000})
    tc.subscribe([topik], on_assign=saat_dapat_partisi)
    pm = PemantauResource().mulai()
    n_tugas, usai, lanjut = 0, False, None
    batas = time.time() + d.get("batas", 3600)                 # pengaman: jangan menunggu SELESAI selamanya
    try:
        while not usai and time.time() < batas:
            m = tc.poll(0.2)
            c = ctl.poll(0)                                        # cek perintah SELESAI dari Master
            if c is not None and not c.error():
                x = json.loads(c.value())
                if x.get("aksi") == "SELESAI" and x.get("eid") == eid:
                    usai = True
                elif x.get("aksi") in ("MULAI", "MIKRO"):          # Master sudah pindah kerja: percobaan ini dibatalkan
                    print(f"Percobaan {eid} dihentikan: Master memulai {x.get('eid')}")
                    lanjut, usai = x, True
            if m is None or m.error():
                continue
            t_terima = time.time()
            tg = json.loads(m.value())
            t_mulai = time.time(); p0 = time.perf_counter()
            baris = beban_kerja(tg)
            layanan = time.perf_counter() - p0
            kirim(TOPIC_BENCH, HOST, {"jenis": "tugas", "run": d["run"], "eid": eid, "tugas": tg["tugas"],
                                      "worker": HOST, "nomor": NOMOR_WORKER, "partisi": m.partition(), "offset": m.offset(),
                                      "t_dibuat": tg["dibuat"], "t_terima": t_terima, "t_mulai": t_mulai,
                                      "t_selesai": time.time(), "detik_layanan": round(layanan, 4),
                                      "baris": baris, "waktu": time.time()})
            n_tugas += 1
            try:
                tc.commit(m, asynchronous=False)
            except KafkaException:
                pass                                              # rebalance: aman
    finally:
        res = pm.berhenti()
        kirim(TOPIC_BENCH, HOST, {"jenis": "resource", "run": d["run"], "eid": eid, "worker": HOST,
                                  "nomor": NOMOR_WORKER, "tugas_selesai": n_tugas, "res": res, "waktu": time.time()})
        produser.flush(10)
        tc.close()
    print(f"Percobaan {eid}: {n_tugas} tugas dikerjakan")
    return lanjut                                              # perintah baru yang sempat terbaca (jika ada)

def mikrobenchmark(d):
    """Ukur latency produce->consume ke broker (jam sama, tanpa masalah selisih jam) dan throughput produser."""
    topik, me = d["topik"], f"{HOST}-{uuid.uuid4().hex[:6]}"
    c = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "mikro-" + me, "auto.offset.reset": "latest",
                  "enable.auto.commit": False})
    siap = {"ok": False}
    c.subscribe([topik], on_assign=lambda cc, p: siap.update(ok=True))
    t0 = time.time()
    while not siap["ok"] and time.time() - t0 < 30:
        c.poll(0.2)
    hasil = []

    def latency(varian, konf, n):
        p = Producer({"bootstrap.servers": BOOTSTRAP, **konf})
        lat = []
        for i in range(n):
            t = time.perf_counter()
            p.produce(topik, key=me, value=json.dumps({"id": me, "v": varian, "i": i}).encode())
            p.poll(0)
            while True:
                m = c.poll(1.0)
                if m is not None and not m.error():
                    try:
                        v = json.loads(m.value())
                    except ValueError:                     # pesan milik Worker lain (mode serempak): lewati
                        v = {}
                    if v.get("id") == me and v.get("v") == varian and v.get("i") == i:
                        break
                if time.perf_counter() - t > 10:           # batas waktu per pesan
                    break
            lat.append((time.perf_counter() - t) * 1000)
        p.flush(5)
        hasil.append({"varian": varian, "p50_ms": round(persentil(lat, 50), 2), "p95_ms": round(persentil(lat, 95), 2),
                      "p99_ms": round(persentil(lat, 99), 2), "rata_ms": round(sum(lat) / len(lat), 2), "throughput": None})

    def lewat(varian, konf, n, ukuran=1024):
        p = Producer({"bootstrap.servers": BOOTSTRAP, **konf})
        isi = (("x" * 64 + "|") * (ukuran // 65 + 1))[:ukuran].encode()    # teks berulang: bisa dimampatkan
        t = time.perf_counter()
        for i in range(n):
            p.produce(topik, key=me, value=isi)
            p.poll(0)
        p.flush(60)
        dur = time.perf_counter() - t
        hasil.append({"varian": varian, "p50_ms": None, "p95_ms": None, "p99_ms": None, "rata_ms": None,
                      "throughput": round(n / dur, 1)})

    n_lat, n_thr = d.get("N_lat", 100), d.get("N_thr", 2000)
    latency("L1 acks=1, linger=0", {"acks": "1", "linger.ms": 0}, n_lat)
    latency("L2 acks=all, linger=0", {"acks": "all", "linger.ms": 0}, n_lat)
    latency("L3 acks=all, linger=50", {"acks": "all", "linger.ms": 50}, n_lat)
    lewat("T1 1KB tanpa kompresi (msg/dtk)", {"acks": "all", "linger.ms": 20, "compression.type": "none"}, n_thr)
    lewat("T2 1KB gzip (msg/dtk)", {"acks": "all", "linger.ms": 20, "compression.type": "gzip"}, n_thr)
    c.close()
    for h in hasil:
        kirim(TOPIC_BENCH, HOST, {"jenis": "mikro", "run": d["run"], "eid": d["eid"], "mode": d["mode"],
                                  "worker": HOST, "nomor": NOMOR_WORKER, **h, "waktu": time.time()})
    produser.flush(10)
    print(f"Mikrobenchmark ({d['mode']}) selesai: {len(hasil)} varian")

def tangani_kontrol(x):
    """Dipanggil dari loop utama (Sel 8) untuk setiap perintah Master."""
    while x:
        sambung = None
        if x.get("aksi") == "MULAI" and NOMOR_WORKER <= x["W"]:
            print(f"Ikut percobaan {x['eid']} (W={x['W']}, P={x['P']}, N={x['N']})")
            sambung = ikut_eksperimen(x, ctl)
        elif x.get("aksi") == "MIKRO" and x.get("untuk") in (0, NOMOR_WORKER):
            print(f"Mikrobenchmark {x['mode']} ...")
            mikrobenchmark(x)
        x = sambung

print("Fungsi benchmark siap.")

## Sel 8. Loop Worker: tugas dari Kafka + perintah benchmark dari Master
Sel ini terus berjalan. Hentikan dengan Interrupt hanya setelah semua selesai.

In [ ]:
# ===== SEL 8: Loop Worker - tugas dari Kafka + perintah benchmark dari Master =====
konsumen = Consumer({
    "bootstrap.servers": BOOTSTRAP,
    "group.id": GROUP_ID,
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,            # commit manual SETELAH tugas selesai
    "max.poll.interval.ms": 1800000,        # tugas boleh lama (30 menit)
    "session.timeout.ms": 45000,
})
konsumen.subscribe([TOPIC_TUGAS], on_assign=lapor_siap)

# Perintah benchmark: hanya pesan BARU (latest), Master menunggu BENCH_SIAP sebelum mengirim perintah
ctl = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": f"{KELOMPOK}-ctl-{HOST}-{uuid.uuid4().hex[:6]}",
                "auto.offset.reset": "latest", "enable.auto.commit": False})
ctl.subscribe([TOPIC_KONTROL], on_assign=lambda c, p: kirim(
    TOPIC_STATUS, HOST, {"run": "-", "worker": HOST, "status": "BENCH_SIAP", "waktu": time.time()}))

print(f"Worker {HOST} menunggu tugas dari '{TOPIC_TUGAS}' dan perintah dari '{TOPIC_KONTROL}' ... (Interrupt untuk berhenti)")

terakhir_hidup = 0
try:
    while True:
        msg = konsumen.poll(0.2)
        if time.time() - terakhir_hidup > 10:      # detak "saya masih hidup" (juga dipakai menaksir selisih jam)
            kirim(TOPIC_STATUS, HOST, {"run": "-", "worker": HOST, "status": "HIDUP", "waktu": time.time()})
            terakhir_hidup = time.time()
        k = ctl.poll(0)                            # perintah benchmark dari Master
        if k is not None and not k.error():
            try:
                tangani_kontrol(json.loads(k.value()))
            except Exception as e:
                print("Perintah benchmark gagal:", e)
        if msg is None:
            continue
        t_terima = time.time()
        if msg.error():
            if msg.error().code() != KafkaError._PARTITION_EOF:
                print("Kafka:", msg.error())
            continue
        try:
            tugas = json.loads(msg.value())
        except ValueError:
            print("Pesan rusak dilewati, offset", msg.offset())
            konsumen.commit(msg, asynchronous=False)
            continue

        umur = time.time() - tugas.get("dibuat", time.time())
        if MAKS_UMUR_TUGAS is not None and umur > MAKS_UMUR_TUGAS:
            print(f"Tugas {tugas['tugas']} dilewati (sisa percobaan lama, umur {umur:,.0f} dtk)")
        else:
            print(f"Menerima tugas {tugas['tugas']} [{tugas.get('kota', 'performa')}] "
                  f"(partisi {msg.partition()}, offset {msg.offset()})")
            kerjakan(tugas, t_terima)
        try:
            konsumen.commit(msg, asynchronous=False)
        except KafkaException as e:
            print("Commit gagal (rebalance), lanjut:", e)   # aman: Master membuang duplikat
except KeyboardInterrupt:
    print("\nWorker dihentikan.")
finally:
    try:
        kirim(TOPIC_STATUS, HOST, {"run": "-", "worker": HOST, "status": "BERHENTI", "waktu": time.time()})
        produser.flush(10)
    finally:
        konsumen.close(); ctl.close()